# Défi — Reranking sans serveur avec Pinecone

Ce notebook construit un pipeline complet de **reranking** (réordonnancement) avec Pinecone :
1. Reranking de documents de base (Apple entreprise vs fruit)
2. Création d'un index **serverless** pour des notes médicales
3. Chargement de données d'exemple (JSONL)
4. Upsert des données dans l'index
5. Fonction d'embedding + recherche sémantique
6. Affichage puis **reranking** des notes cliniques

> ⚠️ **Prérequis :** un compte Pinecone et une clé API (https://app.pinecone.io → *API Keys*).
> À exécuter de préférence sur **Google Colab**.

## Partie 1 — Charger les documents et exécuter le modèle de reranking

### 1. Installer les bibliothèques Pinecone
💡 En cas de conflit de versions, redémarrez le runtime après l'installation.

In [ ]:
!pip install -U pinecone==6.0.1 pinecone-notebooks

### 2. S'authentifier auprès de Pinecone

In [ ]:
import os
if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

### 3. Instancier le client Pinecone

In [ ]:
from pinecone import Pinecone
api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)

### 4. Définir la requête et les documents

On mélange volontairement des documents sur **Apple (l'entreprise)** et **apple (le fruit)**
pour tester la désambiguïsation contextuelle du reranker.

In [ ]:
query = "Tell me about Apple's products"
documents = [
    "An apple is a sweet, edible fruit produced by an apple tree; it is rich in fiber and vitamin C.",          # fruit
    "Apple Inc. designs and sells the iPhone, iPad, Mac computers, Apple Watch and AirPods.",                   # entreprise
    "Apples come in many varieties such as Fuji, Gala and Granny Smith, and are often used to make cider.",     # fruit
    "Apple's latest products include the iPhone 15 Pro and the M3-powered MacBook Pro laptops.",                # entreprise
    "Eating an apple a day is associated with several health benefits, according to popular folklore.",         # fruit/santé (choix libre)
]

### 5. Appeler le service de reranking

In [ ]:
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3,
)

### 6. Examiner les résultats reclassés

L'objet renvoyé expose ses résultats via l'attribut **`.data`** ; chaque élément a un
`.score` et un `.document.text`. Plus le score est élevé, plus le document est pertinent.

In [ ]:
def show_reranked_results(query, matches):
    print(f"Query: {query}\n")
    for i, m in enumerate(matches):
        print(f"{i+1}. score={m.score:.4f}")
        print(f"   {m.document.text}\n")

show_reranked_results(query, reranked.data)

On s'attend à voir les documents sur **Apple l'entreprise** remonter en tête, car la requête
porte sur « Apple's products ».

## Partie 2 — Mettre en place un index serverless pour les notes médicales

### 1. Installer les bibliothèques de données et de modèles
💡 Le téléchargement peut prendre quelques minutes.

In [ ]:
!pip install pandas torch transformers

### 2. Importer les modules et définir les paramètres d'environnement

In [ ]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Paramètres cloud/région (valeurs par défaut valables pour la plupart des comptes)
cloud = os.getenv('PINECONE_CLOUD', 'aws')
region = os.getenv('PINECONE_REGION', 'us-east-1')

# Spécification serverless
spec = ServerlessSpec(cloud=cloud, region=region)

# Nom de l'index
index_name = 'medical-notes-index'

### 3. Créer (ou recréer) l'index

- **`dimension=384`** : correspond au modèle `all-MiniLM-L6-v2`.
- **`metric='cosine'`** : adaptée aux embeddings de texte.

In [ ]:
# Supprimer un éventuel index existant du même nom
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

# Créer un nouvel index serverless
pc.create_index(
    name=index_name,
    dimension=384,
    metric='cosine',
    spec=spec,
)
print(f"Index '{index_name}' créé.")

## Partie 3 — Charger les données d'exemple

### 1. Télécharger et lire le fichier JSONL

> ⚠️ Utilisez bien l'URL GitHub **raw**. L'énoncé pointe vers `refs/heads/master`, mais le
> fichier se trouve désormais sur la branche `main` du dépôt `pinecone-io/examples` — on
> utilise donc l'URL `main` (fonctionnelle).

In [ ]:
import requests
import tempfile

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # URL GitHub raw (branche main)
    url = "https://raw.githubusercontent.com/pinecone-io/examples/main/docs/data/sample_notes_data.jsonl"
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)

### 2. Prévisualiser le DataFrame

`df.shape` renvoie le couple **(lignes, colonnes)**.

In [ ]:
print("Data shape:", df.shape)   # (lignes, colonnes)
df.head()

Le DataFrame contient les colonnes attendues : **`id`**, **`values`** (vecteurs 384-D
pré-calculés) et **`metadata`** (champs cliniques). C'est exactement le format attendu par
`upsert_from_dataframe`.

## Partie 4 — Upsert des données dans l'index

### 1. Instancier le client d'index et faire l'upsert

In [ ]:
index = pc.Index(name=index_name)

# Upsert depuis le DataFrame (colonnes id / values / metadata)
index.upsert_from_dataframe(df)

### 2. Attendre que l'index soit prêt

On attend qu'au moins **1** vecteur (`> 0`) soit indexé avant d'interroger.

In [ ]:
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print("Vector count: ", vector_count)
    return vector_count > 0

while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
index.describe_index_stats()

## Partie 5 — Fonction d'embedding et de requête

### 1. Définir la fonction d'embedding

On fait la moyenne sur **`dim=0`** : `last_hidden_state[0]` a la forme
`(longueur_séquence, 384)`, donc moyenner sur l'axe 0 (les tokens) donne **un seul vecteur
384-D** par entrée (mean pooling).

In [ ]:
def get_embedding(input_question):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**encoded_input)
    embedding = model_output.last_hidden_state[0].mean(dim=0)   # moyenne sur la longueur de séquence
    return embedding

### 2. Exécuter une requête de recherche sémantique

In [ ]:
# Question médicale
question = "patient with chest pain"
query = get_embedding(question).tolist()

# Récupérer les résultats
results = index.query(vector=[query], top_k=10, include_metadata=True)

# Trier par score décroissant
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

## Partie 6 — Afficher et réordonner les notes cliniques

### 1. Afficher les premiers résultats

Dans un match Pinecone, le score est dans **`score`** et les métadonnées dans **`metadata`**.

In [ ]:
def show_results(question, matches):
    print(f"Question: '{question}'")
    print('\nResults:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f'      Score: {match["score"]}')
        print(f'      Metadata: {match["metadata"]}')
        print('')

show_results(question, sorted_matches)

### 2. Préparer les documents pour le reranking

On concatène les métadonnées de chaque note dans un champ texte `reranking_field`.

In [ ]:
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]
transformed_documents[:3]

### 3. Exécuter le reranking serverless

On passe une requête clinique **plus précise** et on indique le champ à classer
(`rank_fields=["reranking_field"]`).

In [ ]:
refined_query = "patient requiring knee surgery"

reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,
    return_documents=True,
)

### 4. Afficher les résultats reclassés

Résultats dans **`.data`** ; pour chaque élément : `.score` et `.document.reranking_field`.

In [ ]:
def show_reranked_results(question, matches):
    print(f"Question: '{question}'")
    print('\nReranked Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f'      Score: {match.score}')
        print(f'      Reranking Field: {match.document.reranking_field}')
        print('')

show_reranked_results(refined_query, reranked_results.data)

**Comparaison :** le reranker réévalue la pertinence des notes vis-à-vis de la requête
affinée, ce qui peut faire remonter des notes plus pertinentes cliniquement que le simple
classement par similarité cosinus de la recherche initiale.

### 5. Nettoyage (facultatif)

Supprime l'index pour éviter des coûts inutiles.

In [ ]:
pc.delete_index(name=index_name)
print("Index supprimé.")

## ✅ Récapitulatif
- Authentification Pinecone et reranking de base (Apple entreprise vs fruit).
- Index serverless (dimension 384, métrique cosinus) créé et alimenté avec 100 notes médicales.
- Recherche sémantique via embeddings `all-MiniLM-L6-v2`.
- Reranking des notes cliniques avec `bge-reranker-v2-m3` et comparaison avec la recherche initiale.

**Idées d'extension :** varier `top_k` / `top_n`, tester d'autres requêtes (« diabetes treatment plan »,
« fracture treatment »), et filtrer par métadonnée lors de la requête.